#### Let's implement a tiny SLR (1) LR parser generator in Python and demo it.
#### It will tokenize simple arithmetic (id/num, +,-,*,/, parentheses), buile SLR(1) tables
#### and parse to an AST while reducing.

In [1]:
import re
from dataclasses import dataclass
from typing import List, Tuple, Dict, Set, FrozenSet, Optional, Any

#### First, we will make lexer.

In [2]:
@dataclass
class Token:
    typ: str
    val: str
    pos: int

def lex(s:str) -> List[Token]:
    token_spec = [
        ("NUM", r'\d+(?:\.\d+)?'),
        ("ID", r'[A-za-z_]\w*'),
        ("PLUS",  r'\+'),
        ("MINUS", r'-'),
        ("MUL",   r'\*'),
        ("DIV",   r'/'),
        ("LP",    r'\('),
        ("RP",    r'\)'),
        ("WS",    r'\s+')
    ]

    token_regex = "|".join(f"(?P<{n}>{r})" for n, r, in token_spec)
    get_token = re.compile(token_regex).match

    pos = 0
    tokens: List[Token] = []
    m = get_token(s, pos)
    while m is not None:
        typ = m.lastgroup
        val = m.group()
        if typ != "WS":
            # Map symbolic types to the terminals we will use
            if type in ("PLUS", "MINUS", "MUL", "DIV", "LP", "RP"):
                # keep symbol names as literal tokens for parse table
                sym_map = {"PLUS":"+","MINUS":"-","MUL":"*","DIV":"/","LP":"(","RP":")"}
                tokens.append(Token(sym_map[typ], val, pos))
            else:
                tokens.append(Token(typ, val, pos))
        pos = m.end()
        m = get_token(s, pos)
    if pos != len(s):
        raise SyntaxError(f"Unexpected character {s[pos]!r} at {pos}")
    tokens.append(Token("$", "$", -1)) # EOF
    return tokens

In [3]:
lex("a + ( b * c - (d - f) * 2 ) * g / h + 52")

[Token(typ='ID', val='a', pos=0),
 Token(typ='PLUS', val='+', pos=2),
 Token(typ='LP', val='(', pos=4),
 Token(typ='ID', val='b', pos=6),
 Token(typ='MUL', val='*', pos=8),
 Token(typ='ID', val='c', pos=10),
 Token(typ='MINUS', val='-', pos=12),
 Token(typ='LP', val='(', pos=14),
 Token(typ='ID', val='d', pos=15),
 Token(typ='MINUS', val='-', pos=17),
 Token(typ='ID', val='f', pos=19),
 Token(typ='RP', val=')', pos=20),
 Token(typ='MUL', val='*', pos=22),
 Token(typ='NUM', val='2', pos=24),
 Token(typ='RP', val=')', pos=26),
 Token(typ='MUL', val='*', pos=28),
 Token(typ='ID', val='g', pos=30),
 Token(typ='DIV', val='/', pos=32),
 Token(typ='ID', val='h', pos=34),
 Token(typ='PLUS', val='+', pos=36),
 Token(typ='NUM', val='52', pos=38),
 Token(typ='$', val='$', pos=-1)]

#### We will use for grammar.

In [4]:
@dataclass(frozen=True)
class Production:
    lhs: str
    rhs: Tuple[str, ...]
    idx: int # production index (for referencing in parse table)

class Grammar:
    def __init__(self, start: str, prods: List[Tuple[str, List[List[str]]]]):
        # prods: list of (lhs, list of rhs alternatives), rhs are list of symbols
        self.start = start
        self.nonterminals: Set[str] = set(lhs for lhs, _ in prods)
        self.productions: List[Production] = [] # The list of grammar rules ex) Production(lhs='E', rhs=('E', '+', 'T'), idx=0)
        idx = 0
        for lhs, alts in prods:
            for alt in alts:
                self.productions.append(Production(lhs, tuple(alt), idx))
                idx += 1

        # compute terminals
        rhs_symbols = set(sym for _, alts in prods for alt in alts for sym in alt)
        self.terminals : Set[str] = set(sym for sym in rhs_symbols if sym not in self.nonterminals)

        # examples 
        # rhs_symbols : {'E', '(', ')', '/', 'T', 'NUM', 'F', '*', 'ID', '-', '+'}
        # self.terminals : {'(', ')', '/', 'NUM', '*', 'ID', '-', '+'}

        # augmented start
        self.aug_start = self.start+"'"
        while self.aug_start in self.nonterminals or self.aug_start in self.terminals:
            self.aug_start += "'"

        # add augmented production at index -1 for clarity later; but we will prepend it to list
        self.aug_prod = Production(self.aug_start, (self.start,), -1) # ex ) self.aug_prod : Production(lhs="E'", rhs=('E',), idx=-1)
        # Re-index productions with augmented as 0 for easier referencing
        self.productions = [Production(self.aug_prod.lhs, self.aug_prod.rhs, 0)] + \
                            [Production(p.lhs, p.rhs, i + 1) for i, p in enumerate(self.productions)]
        
        # Update counts
        self.prod_by_lhs : Dict[str, List[Production]] = {}

        for p in self.productions:
            self.prod_by_lhs.setdefault(p.lhs, []).append(p)

        # print(f"self.prod_by_lhs : {self.prod_by_lhs}")
        # self.prod_by_lhs : {
        #                       "E'": [Production(lhs="E'", rhs=('E',), idx=0)
        #                           ], 
        #                       'E': [Production(lhs='E', rhs=('E', '+', 'T'), idx=1), 
        #                               Production(lhs='E', rhs=('E', '-', 'T'), idx=2), 
        #                               Production(lhs='E', rhs=('T',), idx=3)
        #                           ], 
        #                       'T': [Production(lhs='T', rhs=('T', '*', 'F'), idx=4), 
        #                               Production(lhs='T', rhs=('T', '/', 'F'), idx=5), 
        #                               Production(lhs='T', rhs=('F',), idx=6)
        #                           ], 
        #                       'F': [Production(lhs='F', rhs=('(', 'E', ')'), idx=7), 
        #                               Production(lhs='F', rhs=('ID',), idx=8), 
        #                               Production(lhs='F', rhs=('NUM',), idx=9)
        #                             ]
        #                     }

    def __repr__(self):
        lines = []
        for p in self.productions:
            lines.append(f"{p.idx}: {p.lhs} -> {' '.join(p.rhs)}")
        return "\n".join(lines)
    

G = Grammar(
    start="E",
    prods=[
        ("E", [["E","+","T"], ["E","-","T"], ["T"]]),
        ("T", [["T","*","F"], ["T","/","F"], ["F"]]),
        ("F", [["(","E",")"], ["ID"], ["NUM"]]),
    ]
)

#### First & Follow (for SLR)

In [5]:
def compute_first(grammar: Grammar) -> Dict[str, Set[str]]:
    FIRST: Dict[str, Set[str]] = {sym: set() for sym in grammar.nonterminals | grammar.terminals}
    # print(f"First 1 : {FIRST}") # First 1 : {'E': set(), '*': set(), 'F': set(), '-': set(), ')': set(), 'ID': set(), '/': set(), '(': set(), '+': set(), 'NUM': set(), 'T': set()}

    for t in grammar.terminals: # terminals : First(terminal) = {terminal}
        FIRST[t].add(t) 
    # print(f"First 2 : {FIRST}") # First 2 : {'E': set(), '*': {'*'}, 'F': set(), '-': {'-'}, ')': {')'}, 'ID': {'ID'}, '/': {'/'}, '(': {'('}, '+': {'+'}, 'NUM': {'NUM'}, 'T': set()}

    changed = True
    while changed:
        changed = False
        for p in grammar.productions[1:]: # skip augmented at 0
            X = p.lhs
            rhs = p.rhs
            # no epsilon rules in this grammar; we only add FIRST of first symbol
            if not rhs:
                continue
            # print(f"x : {X}, rhs : {rhs}") # ex) x : E, rhs : ('E', '+', 'T')
            first_rhs = set(FIRST[rhs[0]])
            before = len(FIRST[X])
            # print(f"first_rhs : {first_rhs}, before : {before}")
            FIRST[X] |= first_rhs
            if len(FIRST[X]) != before:
                changed = True

    # In loop
    # x : E, rhs : ('E', '+', 'T')
    # first_rhs : set(), before : 0
    # x : E, rhs : ('E', '-', 'T')
    # first_rhs : set(), before : 0
    # x : E, rhs : ('T',)
    # first_rhs : set(), before : 0
    # x : T, rhs : ('T', '*', 'F')
    # first_rhs : set(), before : 0
    # x : T, rhs : ('T', '/', 'F')
    # first_rhs : set(), before : 0
    # x : T, rhs : ('F',)
    # first_rhs : set(), before : 0
    # x : F, rhs : ('(', 'E', ')')
    # first_rhs : {'('}, before : 0
    # x : F, rhs : ('ID',)
    # first_rhs : {'ID'}, before : 1
    # x : F, rhs : ('NUM',)
    # first_rhs : {'NUM'}, before : 2
    # x : E, rhs : ('E', '+', 'T')                
    return FIRST

def compute_follow(grammar: Grammar, FIRST: Dict[str, Set[str]]) -> Dict[str, Set[str]]:
    FOLLOW: Dict[str, Set[str]] = {A: set() for A in grammar.nonterminals | {grammar.aug_start}}
    FOLLOW[grammar.start].add("$")
    # print(f"FOLLOW 1 : {FOLLOW}") # FOLLOW 1 : {'E': {'$'}, "E'": set(), 'T': set(), 'F': set()}
    changed = True
    while changed:
        changed = False
        for p in grammar.productions[1:]: # skip augmented at 0
            lhs = p.lhs
            rhs = p.rhs
            # print(f"lhs: {lhs}, rhs : {rhs}")
            for i, B in enumerate(rhs):
                if B in grammar.nonterminals:
                    # beta is rhs[i + 1:]
                    beta = rhs[i+1:]
                    before = len(FOLLOW[B])
                    # print(f"beta: {beta}, before : {before}")
                    if beta:
                        FIRST_beta = FIRST[beta[0]]
                        # add FIRST(beta) - {epsilon}; no epsilon in our grammar
                        FOLLOW[B] |= (FIRST_beta - set()) # no epsilon concerns
                    else:
                        FOLLOW[B] |= FOLLOW[lhs]
                    if len(FOLLOW[B]) != before:
                        changed = True

        # In loop 
        # lhs: E, rhs : ('E', '+', 'T')
        # beta: ('+', 'T'), before : 1
        # beta: (), before : 0
        # lhs: E, rhs : ('E', '-', 'T')
        # beta: ('-', 'T'), before : 2
        # beta: (), before : 2
        # lhs: E, rhs : ('T',)
        # beta: (), before : 3
        # lhs: T, rhs : ('T', '*', 'F')
        # beta: ('*', 'F'), before : 3
        # beta: (), before : 0
        # lhs: T, rhs : ('T', '/', 'F')
        # beta: ('/', 'F'), before : 4
        # beta: (), before : 4
        # lhs: T, rhs : ('F',)
        # beta: (), before : 5
        # lhs: F, rhs : ('(', 'E', ')')
        # beta: (')',), before : 3
        # lhs: F, rhs : ('ID',)
        # lhs: F, rhs : ('NUM',)
        # lhs: E, rhs : ('E', '+', 'T')
        # beta: ('+', 'T'), before : 4
        # beta: (), before : 5
        # lhs: E, rhs : ('E', '-', 'T')                        
    return FOLLOW

FIRST = compute_first(G)
FOLLOW = compute_follow(G, FIRST)

(FIRST, FOLLOW)

# FIRST : {'E': {'(', 'ID', 'NUM'},
#            '*': {'*'},
#            'F': {'(', 'ID', 'NUM'},
#            '-': {'-'},
#            ')': {')'},
#           'ID': {'ID'},
#            '/': {'/'},
#           '(': {'('},
#            '+': {'+'},
#            'NUM': {'NUM'},
#            'T': {'(', 'ID', 'NUM'}
#       }

# FOLLOW :  {'E': {'$', ')', '+', '-'},
#            "E'": set(),
#            'T': {'$', ')', '*', '+', '-', '/'},
#            'F': {'$', ')', '*', '+', '-', '/'}
#        }

({')': {')'},
  '*': {'*'},
  '/': {'/'},
  'ID': {'ID'},
  'T': {'(', 'ID', 'NUM'},
  '+': {'+'},
  '(': {'('},
  'NUM': {'NUM'},
  'E': {'(', 'ID', 'NUM'},
  'F': {'(', 'ID', 'NUM'},
  '-': {'-'}},
 {'F': {'$', ')', '*', '+', '-', '/'},
  "E'": set(),
  'E': {'$', ')', '+', '-'},
  'T': {'$', ')', '*', '+', '-', '/'}})

#### ---- LR(0) Items / Canonical Collection ----

In [17]:
@dataclass(frozen=True)
class Item:
    prod_idx: int
    dot: int #position of dot in rhs (0..len)

def closure(grammar: Grammar, items: Set[Item]) -> Set[Item]:
    changed = True
    I = set(items)
    while changed:
        changed = False
        # for each [A -> α • B β] add [B -> • γ] for all B->γ
        for it in list(I):
            p = grammar.productions[it.prod_idx]
            if it.dot < len(p.rhs):
                B = p.rhs[it.dot]
                if B in grammar.nonterminals:
                    for pB in grammar.prod_by_lhs[B]:
                        itemB = Item(pB.idx, 0)
                        if itemB not in I:
                            I.add(itemB)
                            changed = True
    print(f"return value : {I}")
    return I

def goto(grammar: Grammar, I:Set[Item], X:str)->Set[Item]:
    J = set()
    for it in I:
        p = grammar.productions[it.prod_idx]
        if it.dot < len(p.rhs) and p.rhs[it.dot] == X:
            J.add(Item(it.prod_idx, it.dot+1))
        print(f"J : {J}, p : {p}")
        
    return closure(grammar, J) if J else set()

def canonical_collection(grammar: Grammar) -> Tuple[List[Set[Item]], Dict[Tuple[int,str], int]]:
    symbols = list(grammar.terminals | grammar.nonterminals | {"$"}) # include $ for completeness
    C: List[Set[Item]] = []
    trans: Dict[Tuple[int, str], int] = {} # (state, symbol) -> state
    I0 = closure(grammar, {Item(0, 0)})  # [S' -> • E]
    C.append(I0)
    work = [0]
    seen: Dict[FrozenSet[Item], int] = {frozenset(I0): 0}
    while work:
        i = work.pop()
        I = C[i]
        # only consider real grammar symbols (terminals + nonterminals), not $
        for X in grammar.terminals | grammar.nonterminals:
            J = goto(grammar, I, X)
            if not J:
                continue
            fJ = frozenset(J)
            if fJ not in seen:
                seen[fJ] = len(C)
                C.append(J)
                work.append(seen[fJ])
            trans[(i,X)] = seen[fJ]
        print(f"C : {C}, trans : {trans}")
    return C, trans

C, trans = canonical_collection(G)

return value : {Item(prod_idx=9, dot=0), Item(prod_idx=4, dot=0), Item(prod_idx=0, dot=0), Item(prod_idx=7, dot=0), Item(prod_idx=2, dot=0), Item(prod_idx=8, dot=0), Item(prod_idx=3, dot=0), Item(prod_idx=5, dot=0), Item(prod_idx=6, dot=0), Item(prod_idx=1, dot=0)}
J : set(), p : Production(lhs='F', rhs=('NUM',), idx=9)
J : set(), p : Production(lhs='T', rhs=('T', '*', 'F'), idx=4)
J : set(), p : Production(lhs="E'", rhs=('E',), idx=0)
J : set(), p : Production(lhs='F', rhs=('(', 'E', ')'), idx=7)
J : set(), p : Production(lhs='E', rhs=('E', '-', 'T'), idx=2)
J : set(), p : Production(lhs='F', rhs=('ID',), idx=8)
J : set(), p : Production(lhs='E', rhs=('T',), idx=3)
J : set(), p : Production(lhs='T', rhs=('T', '/', 'F'), idx=5)
J : set(), p : Production(lhs='T', rhs=('F',), idx=6)
J : set(), p : Production(lhs='E', rhs=('E', '+', 'T'), idx=1)
J : set(), p : Production(lhs='F', rhs=('NUM',), idx=9)
J : set(), p : Production(lhs='T', rhs=('T', '*', 'F'), idx=4)
J : set(), p : Production(

#### ---- Build SLR(1) Parse Tables ----

In [19]:
# ACTION: (state, terminal) -> ('s', j) | ('r', prod_idx) | ('acc',)
# GOTO:   (state, nonterminal) -> j

Action: Dict[Tuple[int, str], Tuple[str, int]] = {}
Goto: Dict[Tuple[int, str], int] = {}

def build_slr_tables(grammar: Grammar) -> Tuple[Dict[Tuple[int, str], Tuple[str, int]], Dict[Tuple[int, str], int]]:
    ACTION: Dict[Tuple[int, str], Tuple[str, int]] = {}
    GOTO: Dict[Tuple[int, str], int] = {}
    terminals = set(grammar.terminals) | {"$"}

    for i, I in enumerate(C):
        # 1) shifts
        for a in grammar.terminals:
            J = goto(grammar, I, a)
            if J:
                j = None
                for idx, st in enumerate(C):
                    if st == J:
                        j = idx
                        break
                if j is None:
                    raise RuntimeError("State not found during shift")
                key = (i, a)
                if key in ACTION:
                    raise RuntimeError(F"Shift conflict at state {i}, symbol {a}")
                ACTION[key] = ("s", j)
        # 2) reductions / accept
        for it in I:
            p = grammar.productions[it.prod_idx]
            if it.dot == len(p.rhs):
                if p.lhs == grammar.aug_start:
                    ACTION[(i, "$")] = ("acc", 0)
                else:
                    for a in FOLLOW[p.lhs]:
                        key = (i, a)
                        if key in ACTION and ACTION[key][0] != "r":
                            # detect conflicts to be explicit
                            raise RuntimeError(f"Conflict at state {i}, on {a}: existing {ACTION[key]}, new reduce {p.idx}")
                        ACTION[key] = ("r", p.idx)
        # 3) GOTO table
        for A in grammar.nonterminals:
            J = goto(grammar, I, A)
            if J:
                j = None
                for idx, st in enumerate(C):
                    if st == J:
                        j = idx
                        break
                if j is None:
                    raise RuntimeError("Goto state not found")
                GOTO[(i, A)] = j
    return ACTION, GOTO

Action, Goto = build_slr_tables(G)

Action, Goto

J : set(), p : Production(lhs='F', rhs=('NUM',), idx=9)
J : set(), p : Production(lhs='T', rhs=('T', '*', 'F'), idx=4)
J : set(), p : Production(lhs="E'", rhs=('E',), idx=0)
J : set(), p : Production(lhs='F', rhs=('(', 'E', ')'), idx=7)
J : set(), p : Production(lhs='E', rhs=('E', '-', 'T'), idx=2)
J : set(), p : Production(lhs='F', rhs=('ID',), idx=8)
J : set(), p : Production(lhs='E', rhs=('T',), idx=3)
J : set(), p : Production(lhs='T', rhs=('T', '/', 'F'), idx=5)
J : set(), p : Production(lhs='T', rhs=('F',), idx=6)
J : set(), p : Production(lhs='E', rhs=('E', '+', 'T'), idx=1)
J : set(), p : Production(lhs='F', rhs=('NUM',), idx=9)
J : set(), p : Production(lhs='T', rhs=('T', '*', 'F'), idx=4)
J : set(), p : Production(lhs="E'", rhs=('E',), idx=0)
J : set(), p : Production(lhs='F', rhs=('(', 'E', ')'), idx=7)
J : set(), p : Production(lhs='E', rhs=('E', '-', 'T'), idx=2)
J : set(), p : Production(lhs='F', rhs=('ID',), idx=8)
J : set(), p : Production(lhs='E', rhs=('T',), idx=3)
J 

({(0, 'ID'): ('s', 1),
  (0, '('): ('s', 3),
  (0, 'NUM'): ('s', 4),
  (1, ')'): ('r', 8),
  (1, '*'): ('r', 8),
  (1, '/'): ('r', 8),
  (1, '$'): ('r', 8),
  (1, '+'): ('r', 8),
  (1, '-'): ('r', 8),
  (2, '*'): ('s', 10),
  (2, '/'): ('s', 11),
  (2, ')'): ('r', 3),
  (2, '$'): ('r', 3),
  (2, '-'): ('r', 3),
  (2, '+'): ('r', 3),
  (3, 'ID'): ('s', 1),
  (3, '('): ('s', 3),
  (3, 'NUM'): ('s', 4),
  (4, ')'): ('r', 9),
  (4, '*'): ('r', 9),
  (4, '/'): ('r', 9),
  (4, '$'): ('r', 9),
  (4, '+'): ('r', 9),
  (4, '-'): ('r', 9),
  (5, '+'): ('s', 7),
  (5, '-'): ('s', 8),
  (5, '$'): ('acc', 0),
  (6, ')'): ('r', 6),
  (6, '*'): ('r', 6),
  (6, '/'): ('r', 6),
  (6, '$'): ('r', 6),
  (6, '+'): ('r', 6),
  (6, '-'): ('r', 6),
  (7, 'ID'): ('s', 1),
  (7, '('): ('s', 3),
  (7, 'NUM'): ('s', 4),
  (8, 'ID'): ('s', 1),
  (8, '('): ('s', 3),
  (8, 'NUM'): ('s', 4),
  (9, '*'): ('s', 10),
  (9, '/'): ('s', 11),
  (9, ')'): ('r', 2),
  (9, '$'): ('r', 2),
  (9, '-'): ('r', 2),
  (9, '+'): ('